### Jessica West asked for variant controls and if most of the variant effects observed were decreasing compared to increasing

In [4]:
import ast
import pandas as pd
import os
import numpy as np
import yaml

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()


# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

In [7]:
def get_gene_name(header, with_controls=True):
    """Returns the gene name: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1 => SKI"""
    if 'cardiac_neuro_cava_random' not in header:
        if not with_controls:
            print(header)
            raise ValueError("Function only defined for tested headers")
        return "Not given - control sequence"


    if 'ALT_' in header:
        return header.split(":ALT_")[1].split('|')[0]

    if 'REF_' in header:
        return header.split(":REF_")[1].split('|')[0]

    return header.split(':')[1].split('|')[0]



from collections import defaultdict


def get_gene_lookup_dict(gene_lists, gene_list_dir='/home/kisa/coding/80K_MPRA/MPRA_design/resources/gene_lists'):
    """
    prepare a dict which has gene_name: (list of associated gene_sets)
    :gene_lists - list of files containing gene names and have useful names
    """
    gene_lookup_dict = defaultdict(list)
    for file in gene_lists:
        with open(os.path.join(gene_list_dir, file), 'r') as f:
            gene_list = f.read().splitlines()
            gene_list_name = file.split('.')[0]
            for gene in gene_list:
                gene_lookup_dict[gene].append(gene_list_name)
    return gene_lookup_dict


In [8]:
mpra_80k_metadata_path = config['files']['creating']['NGN2_variants_metadata_bcalm_2025_bbmap_normalized_counts']

ngn2_80k_variant_metadata_df = pd.read_csv(mpra_80k_metadata_path, sep='\t', compression='gzip', low_memory=False)

ngn2_80k_variant_metadata_df['is_significant'] = ngn2_80k_variant_metadata_df['bcalm_variant_effect_adjusted_p_value'] < 0.1

In [9]:
ngn2_80k_variant_metadata_df = ngn2_80k_variant_metadata_df.loc[ngn2_80k_variant_metadata_df['ID'].str.startswith("cardiac_neuro_cava_random")].copy()
ngn2_80k_variant_metadata_df

,ID,REF,ALT,ref_sequence,alt_sequence,variant_pos,bcalm_variant_effect_data_exists,bcalm_variant_effect_adjusted_p_value,bcalm_variant_effect_log_ratio_activity,bcalm_reference_adjusted_p_value,bcalm_reference_log_ratio_activity,bcalm_alternative_adjusted_p_value,bcalm_alternative_log_ratio_activity,label,is_significant
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,[83],False,NaN,NaN,0.022133,0.131536,5.965362e-07,0.190591,cardiac_neuro_cava_random,False
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,[181],False,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,False
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGAGGC...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGATGC...,[43],False,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,[116],True,0.944787,-0.043372,1.000000,-0.068107,1.000000e+00,-0.072051,cardiac_neuro_cava_random,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,[205],True,0.926799,-0.036392,1.000000,0.001944,1.000000e+00,-0.005889,cardiac_neuro_cava_random,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46369,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,[64],True,0.911154,-0.076406,0.015993,0.100370,4.511542e-01,0.065547,cardiac_neuro_cava_random,False
46370,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,[149],True,0.748684,-0.153448,1.000000,-0.080632,9.174673e-02,-0.090558,cardiac_neuro_cava_random,False
46371,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,[126],True,0.989551,0.007846,0.409652,0.056209,4.362900e-01,0.052674,cardiac_neuro_cava_random,False
46372,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCTC...,[44],True,0.514030,-0.243036,0.409652,0.056209,1.000000e+00,0.016824,cardiac_neuro_cava_random,False


#### Add gene and gene set information

In [10]:
# get the gene id
# get gene_list files:
gene_list_dir = '/home/kisa/coding/80K_MPRA/MPRA_design/resources/gene_lists'
gene_lists = os.listdir(gene_list_dir)

# prepare a dict which has gene_name: (tuple of associated gene_sets)
gene_lookup_dict = get_gene_lookup_dict(gene_lists=gene_lists)

# # get gene name from variant id
ngn2_80k_variant_metadata_df['gene_name'] = ngn2_80k_variant_metadata_df["ALT"].apply(get_gene_name)

ngn2_80k_variant_metadata_df['gene_set'] = ngn2_80k_variant_metadata_df['gene_name'].apply(lambda gene: gene_lookup_dict[gene])

#### Load and add the gnomad info (AF, AC)

In [11]:
gnomad_data_all_variants = pd.read_csv(config['files']['creating']['gnomad_data_all_variants_local'], sep="\t") # 46374
# add gnomad prefix to all the columns
gnomad_data_all_variants.columns = ['gnomad_' + col for col in gnomad_data_all_variants.columns]

gnomad_data_all_variants.columns

Index(['gnomad_header', 'gnomad_chrom', 'gnomad_pos', 'gnomad_ref',
       'gnomad_alt', 'gnomad_AC', 'gnomad_AF', 'gnomad_AF_popmax',
       'gnomad_AF_eas', 'gnomad_AF_nfe', 'gnomad_AF_fin', 'gnomad_AF_afr',
       'gnomad_AF_asj'],
      dtype='object')

In [12]:
gnomad_data_all_variants

,gnomad_header,gnomad_chrom,gnomad_pos,gnomad_ref,gnomad_alt,gnomad_AC,gnomad_AF,gnomad_AF_popmax,gnomad_AF_eas,gnomad_AF_nfe,gnomad_AF_fin,gnomad_AF_afr,gnomad_AF_asj
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1,2179591,T,C,71917.0,0.472883,0.667391,0.347516,0.393314,0.37656,0.667391,0.374352
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1,2191444,G,A,2617.0,0.017203,0.059656,0.000000,0.000147,0.00000,0.059656,0.000000
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1,2192015,G,T,1.0,0.000007,0.000024,0.000000,0.000000,0.00000,0.000024,0.000000
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1,2192366,T,G,1.0,0.000007,0.000015,0.000000,0.000015,0.00000,0.000000,0.000000
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1,2193142,G,A,7205.0,0.047348,0.161114,0.000000,0.001000,0.00000,0.161114,0.000576
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46369,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X,154545206,A,G,1.0,0.000009,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000
46370,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X,154549923,T,G,1.0,0.000009,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000
46371,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X,154552289,C,T,1.0,0.000009,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000
46372,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X,154552371,G,A,2.0,0.000018,0.000561,0.000561,0.000000,0.00000,0.000000,0.000000


In [13]:
variant_bcalm_metadata_df_gnomad_info = ngn2_80k_variant_metadata_df.merge(gnomad_data_all_variants, left_on="ALT", right_on='gnomad_header', how='inner')

In [14]:
# add is common
variant_bcalm_metadata_df_gnomad_info["is_common_variant"] = variant_bcalm_metadata_df_gnomad_info["gnomad_AF"] > 0.01

# add positive or negative effect direction
variant_bcalm_metadata_df_gnomad_info["is_decreasing"] =  variant_bcalm_metadata_df_gnomad_info["bcalm_variant_effect_log_ratio_activity"] < 0


# add absolute variant effect
variant_bcalm_metadata_df_gnomad_info["abs_variant_effect"] = variant_bcalm_metadata_df_gnomad_info["bcalm_variant_effect_log_ratio_activity"].abs()

# get normal number variant effect
variant_bcalm_metadata_df_gnomad_info['rounded_variant_effect'] = variant_bcalm_metadata_df_gnomad_info["bcalm_variant_effect_log_ratio_activity"].apply(lambda var_effect: round(var_effect, 3))

In [15]:
variant_bcalm_metadata_df_gnomad_info

,ID,REF,ALT,ref_sequence,alt_sequence,variant_pos,bcalm_variant_effect_data_exists,bcalm_variant_effect_adjusted_p_value,bcalm_variant_effect_log_ratio_activity,bcalm_reference_adjusted_p_value,...,gnomad_AF_popmax,gnomad_AF_eas,gnomad_AF_nfe,gnomad_AF_fin,gnomad_AF_afr,gnomad_AF_asj,is_common_variant,is_decreasing,abs_variant_effect,rounded_variant_effect
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,[83],False,NaN,NaN,0.022133,...,0.667391,0.347516,0.393314,0.37656,0.667391,0.374352,True,False,NaN,NaN
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,[181],False,NaN,NaN,NaN,...,0.059656,0.000000,0.000147,0.00000,0.059656,0.000000,True,False,NaN,NaN
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGAGGC...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGATGC...,[43],False,NaN,NaN,NaN,...,0.000024,0.000000,0.000000,0.00000,0.000024,0.000000,False,False,NaN,NaN
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,[116],True,0.944787,-0.043372,1.000000,...,0.000015,0.000000,0.000015,0.00000,0.000000,0.000000,False,True,0.043372,-0.043
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,[205],True,0.926799,-0.036392,1.000000,...,0.161114,0.000000,0.001000,0.00000,0.161114,0.000576,True,True,0.036392,-0.036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46369,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,[64],True,0.911154,-0.076406,0.015993,...,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000,False,True,0.076406,-0.076
46370,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,[149],True,0.748684,-0.153448,1.000000,...,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000,False,True,0.153448,-0.153
46371,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,[126],True,0.989551,0.007846,0.409652,...,0.000019,0.000000,0.000019,0.00000,0.000000,0.000000,False,False,0.007846,0.008
46372,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCTC...,[44],True,0.514030,-0.243036,0.409652,...,0.000561,0.000561,0.000000,0.00000,0.000000,0.000000,False,True,0.243036,-0.243


#### Check if more decreasing in common variants around random genes (significant only 15: 9 decreasing, 6 increasing)

In [16]:
variant_bcalm_metadata_df_gnomad_info_random = variant_bcalm_metadata_df_gnomad_info.loc[variant_bcalm_metadata_df_gnomad_info['gene_set'].apply(lambda gene_set: "random" in gene_set)].copy()

In [17]:
variant_bcalm_metadata_df_gnomad_info_random['is_significant'].value_counts(normalize=True)

is_significant
False    0.970956
True     0.029044
Name: proportion, dtype: float64

In [18]:
variant_bcalm_metadata_df_gnomad_info_random_common = variant_bcalm_metadata_df_gnomad_info_random.loc[variant_bcalm_metadata_df_gnomad_info_random["gnomad_AF"] > 0.01].copy()

In [19]:
variant_bcalm_metadata_df_gnomad_info_random_common['is_significant'].value_counts(normalize=True)
variant_bcalm_metadata_df_gnomad_info_random_common['is_significant'].value_counts()

is_significant
False    585
True      15
Name: count, dtype: int64

In [20]:
variant_bcalm_metadata_df_gnomad_info_random_common_significant = variant_bcalm_metadata_df_gnomad_info_random_common.loc[variant_bcalm_metadata_df_gnomad_info_random_common['is_significant']].copy()
variant_bcalm_metadata_df_gnomad_info_random_common_significant['is_decreasing'].value_counts()

is_decreasing
True     9
False    6
Name: count, dtype: int64

In [21]:
from scipy.stats import binomtest
# total number of variants
n = variant_bcalm_metadata_df_gnomad_info_random_common_significant.shape[0]

k = variant_bcalm_metadata_df_gnomad_info_random_common_significant['is_decreasing'].sum()

p_value = binomtest(k, n, p=0.5, alternative='two-sided')

print(f"Decreasing: {k}/{n}")
print(f"P-value: {p_value}")

Decreasing: 9/15
P-value: BinomTestResult(k=9, n=15, alternative='two-sided', statistic=0.6, pvalue=0.6072387695312499)


In [22]:
# how is this among all significant common variants?
variant_bcalm_metadata_df_gnomad_info_common_sig = variant_bcalm_metadata_df_gnomad_info.loc[(variant_bcalm_metadata_df_gnomad_info['is_common_variant']) & (variant_bcalm_metadata_df_gnomad_info['is_significant'])].copy()
variant_bcalm_metadata_df_gnomad_info_common_sig['is_decreasing'].value_counts(normalize=True)
variant_bcalm_metadata_df_gnomad_info_common_sig['is_decreasing'].value_counts() # 554

is_decreasing
True     311
False    243
Name: count, dtype: int64

In [23]:
# binomial test for significant common variants to be different in increasing decreasing
from scipy.stats import binomtest
# total number of variants
n = variant_bcalm_metadata_df_gnomad_info_common_sig.shape[0]

k = variant_bcalm_metadata_df_gnomad_info_common_sig['is_decreasing'].sum()

p_value = binomtest(k, n, p=0.5, alternative='two-sided')

print(f"Decreasing: {k}/{n}")
print(f"P-value: {p_value}")

Decreasing: 311/554
P-value: BinomTestResult(k=311, n=554, alternative='two-sided', statistic=0.5613718411552346, pvalue=0.004377227501945699)


In [24]:
# what about within the 100 highest effect variants?
variant_bcalm_metadata_df_gnomad_info_common_sig_sorted = variant_bcalm_metadata_df_gnomad_info_common_sig.sort_values(by='abs_variant_effect', ascending=False)

In [25]:
variant_bcalm_metadata_df_gnomad_info_common_sig_sorted_top100 = variant_bcalm_metadata_df_gnomad_info_common_sig_sorted.head(n=100).copy()


In [26]:
variant_bcalm_metadata_df_gnomad_info_common_sig_sorted_top100

# how many common ?
variant_bcalm_metadata_df_gnomad_info_common_sig_sorted_top100['is_decreasing'].value_counts()

is_decreasing
True     52
False    48
Name: count, dtype: int64

In [27]:
# total number of variants
n = variant_bcalm_metadata_df_gnomad_info_common_sig_sorted_top100.shape[0]

k = variant_bcalm_metadata_df_gnomad_info_common_sig_sorted_top100['is_decreasing'].sum()

p_value = binomtest(k, n, p=0.5, alternative='two-sided')

print(f"Decreasing: {k}/{n}")
print(f"P-value: {p_value}")

Decreasing: 52/100
P-value: BinomTestResult(k=52, n=100, alternative='two-sided', statistic=0.52, pvalue=0.7643534344026669)


In [28]:
sig_variant_bcalm_metadata_df_gnomad_info = variant_bcalm_metadata_df_gnomad_info.loc[variant_bcalm_metadata_df_gnomad_info['is_significant']].copy()

In [29]:
sig_variant_bcalm_metadata_df_gnomad_info.shape[0]

1304

In [81]:
# binomial test for significant common variants to be different in increasing decreasing
from scipy.stats import binomtest
# total number of variants
n = sig_variant_bcalm_metadata_df_gnomad_info.shape[0]

k = sig_variant_bcalm_metadata_df_gnomad_info['is_decreasing'].sum()

p_value = binomtest(k, n, p=0.5, alternative='two-sided')

print(f"Decreasing: {k}/{n}")
print(f"P-value: {p_value}")

Decreasing: 669/1304
P-value: BinomTestResult(k=669, n=1304, alternative='two-sided', statistic=0.5130368098159509, pvalue=0.36080050986106565)


In [82]:
sig_variant_bcalm_metadata_df_gnomad_info_not_common = sig_variant_bcalm_metadata_df_gnomad_info.loc[~sig_variant_bcalm_metadata_df_gnomad_info['is_common_variant']]

In [83]:
# binomial test for all significant variants to be different in increasing decreasing
from scipy.stats import binomtest
# total number of variants
n = sig_variant_bcalm_metadata_df_gnomad_info_not_common.shape[0]

k = sig_variant_bcalm_metadata_df_gnomad_info_not_common['is_decreasing'].sum()

p_value = binomtest(k, n, p=0.5, alternative='two-sided')

print(f"Decreasing: {k}/{n}")
print(f"P-value: {p_value}")

Decreasing: 358/750
P-value: BinomTestResult(k=358, n=750, alternative='two-sided', statistic=0.47733333333333333, pvalue=0.22818401438503716)


In [30]:
sig_variant_bcalm_metadata_df_gnomad_info_sorted = sig_variant_bcalm_metadata_df_gnomad_info.sort_values(by="abs_variant_effect", ascending=False)

In [31]:
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100 = sig_variant_bcalm_metadata_df_gnomad_info_sorted.head(n=100).copy()

In [32]:
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100['is_decreasing'].value_counts()

is_decreasing
False    52
True     48
Name: count, dtype: int64

In [33]:
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100['is_common_variant'].value_counts()

is_common_variant
False    69
True     31
Name: count, dtype: int64

In [34]:
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100

,ID,REF,ALT,ref_sequence,alt_sequence,variant_pos,bcalm_variant_effect_data_exists,bcalm_variant_effect_adjusted_p_value,bcalm_variant_effect_log_ratio_activity,bcalm_reference_adjusted_p_value,...,gnomad_AF_popmax,gnomad_AF_eas,gnomad_AF_nfe,gnomad_AF_fin,gnomad_AF_afr,gnomad_AF_asj,is_common_variant,is_decreasing,abs_variant_effect,rounded_variant_effect
26257,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:REF_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,CAGGTTCACTGTTGGGCTCTGATCCCACCTTCCCACCATGGGGACA...,CAGGTTCACTGTTGGGCTCTGATCCCACCTTCCCACCATGGGGACA...,[63],True,1.257254e-50,1.457050,1.000000e+00,...,0.000024,0.000000,0.000000,0.000000,0.000024,0.000000,False,False,1.457050,1.457
34763,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,[164],True,6.388034e-93,1.443816,6.824671e-58,...,0.000015,0.000000,0.000015,0.000000,0.000000,0.000000,False,False,1.443816,1.444
34711,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,ATTAACTGTAGATTGCATGCTGAATTCAGCCTAAAAACAAGTTACA...,ATTAACTGTAGATTGCATGCTGAATTCAGCCTAAAAACAAGTTACA...,[189],True,5.417784e-60,1.389350,1.000000e+00,...,0.026812,0.000771,0.026812,0.010336,0.005196,0.038307,True,False,1.389350,1.389
4130,cardiac_neuro_cava_random:ALT_NOS1AP|ENSG00000...,cardiac_neuro_cava_random:REF_NOS1AP|ENSG00000...,cardiac_neuro_cava_random:ALT_NOS1AP|ENSG00000...,AACTGAAAGACACAGCCTCCCCATGGGCACGCTGGTTGGATCGTTC...,AACTGAAAGACACAGCCTCCCCATGGGCACGCTGGTTGGATCGTTC...,[200],True,3.861277e-22,-1.345649,6.406940e-19,...,0.000207,0.000000,0.000000,0.000000,0.000000,0.000000,False,True,1.345649,-1.346
45310,cardiac_neuro_cava_random:ALT_DMD|ENSG00000198...,cardiac_neuro_cava_random:REF_DMD|ENSG00000198...,cardiac_neuro_cava_random:ALT_DMD|ENSG00000198...,TTTGTTTGTTTGTTTGAGACGGAGTCTTGCTCAGTCGCCCAGGCTG...,TTTGTTTGTTTGTTTGAGACGGAGTCTTGCTCAGTCGCCCAGGCTG...,[151],True,1.835988e-19,1.324817,1.065542e-01,...,0.000019,0.000000,0.000019,0.000000,0.000000,0.000000,False,False,1.324817,1.325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43078,cardiac_neuro_cava_random:ALT_VPS13B|ENSG00000...,cardiac_neuro_cava_random:REF_VPS13B|ENSG00000...,cardiac_neuro_cava_random:ALT_VPS13B|ENSG00000...,TGGCCCCTACTTAGAGAATCAAAGAGGTTATTTATTCAGATGATGA...,TGGCCCCTACTTAGAGAATCAAAGAGGTTATTTATTCAGATGATGA...,[73],True,2.046138e-07,0.740596,1.000000e+00,...,0.000015,0.000000,0.000015,0.000000,0.000000,0.000000,False,False,0.740596,0.741
34764,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,[188],True,4.185262e-34,-0.734095,6.824671e-58,...,0.000015,0.000000,0.000015,0.000000,0.000000,0.000000,False,True,0.734095,-0.734
24005,cardiac_neuro_cava_random:ALT_PNKP|ENSG0000003...,cardiac_neuro_cava_random:REF_PNKP|ENSG0000003...,cardiac_neuro_cava_random:ALT_PNKP|ENSG0000003...,TGAGAAATGGTCCCAACAGGCCTCCTGGGGGAGTCTAGGAGGGATC...,TGAGAAATGGTCCCAACAGGCCTCCTGGGGGAGTCTAGGAGGGATC...,[134],True,1.640227e-16,0.731763,1.000000e+00,...,0.000066,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.731763,0.732
40915,cardiac_neuro_cava_random:ALT_RINT1|ENSG000001...,cardiac_neuro_cava_random:REF_RINT1|ENSG000001...,cardiac_neuro_cava_random:ALT_RINT1|ENSG000001...,GAGGCTTAAGATACTGCTGGTACCGAAGCTGGTACTTGGATACCTG...,GAGGCTTAAGATACTGCTGGTACCGAAGCTGGTACTTGGATACCTG...,[73],True,8.983828e-17,0.731185,2.989492e-01,...,0.000385,0.000385,0.000015,0.000000,0.000000,0.000000,False,False,0.731185,0.731


In [35]:
# get reference and alternative names
sig_variants_highest_effect_names = sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100["REF"].to_list() + sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100["ALT"].to_list()

# get the bcalm variant effects
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100["rounded_variant_effect"].to_list()

[1.457,
 1.444,
 1.389,
 -1.346,
 1.325,
 1.317,
 1.283,
 -1.242,
 -1.151,
 1.143,
 1.136,
 -1.13,
 1.123,
 -1.121,
 1.105,
 1.105,
 -1.102,
 -1.083,
 -1.07,
 1.068,
 -1.048,
 -1.038,
 1.027,
 -1.022,
 -1.011,
 0.999,
 0.998,
 -0.993,
 -0.975,
 -0.973,
 0.969,
 0.949,
 -0.948,
 0.941,
 -0.939,
 -0.928,
 0.928,
 -0.924,
 0.908,
 0.905,
 -0.899,
 -0.899,
 0.893,
 0.89,
 -0.888,
 -0.884,
 -0.88,
 0.878,
 -0.876,
 0.875,
 -0.868,
 0.867,
 -0.862,
 -0.861,
 -0.857,
 -0.857,
 0.845,
 -0.845,
 0.844,
 0.844,
 -0.842,
 0.834,
 -0.834,
 0.831,
 0.821,
 -0.82,
 0.82,
 0.819,
 0.818,
 0.816,
 0.814,
 0.812,
 -0.808,
 0.806,
 0.806,
 -0.799,
 -0.794,
 -0.79,
 -0.788,
 0.783,
 -0.782,
 -0.776,
 0.773,
 0.772,
 -0.772,
 -0.771,
 0.769,
 0.764,
 0.759,
 0.755,
 -0.754,
 -0.75,
 0.75,
 0.748,
 -0.743,
 0.741,
 -0.734,
 0.732,
 0.731,
 -0.724]

In [ ]:
# make variant effect as a normal number and add it to the reference and alternative names
sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100['var_effect_ref_name'] = sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100['REF']

In [ ]:
ngn2_80k_variant_metadata_df.loc[ngn2_80k_variant_metadata_df["bcalm_variant_effect_adjusted_p_value"]]


# get the info if random gene

# annotate the gnomad information

SyntaxError: unexpected EOF while parsing (3985376327.py, line 1)

In [36]:
import pandas as pd
import ast
metadata_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz"
metadata_table = pd.read_csv(metadata_path,sep="\t")

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]


# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_table[col] = metadata_table[col].apply(safe_eval)


In [37]:
# 1) First, melt the variant map to have a flat list of (name, variant_effect) pairs
ref_df = sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100[['REF', 'rounded_variant_effect']].rename(columns={'REF': 'name'})
alt_df = sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100[['ALT', 'rounded_variant_effect']].rename(columns={'ALT': 'name'})
name_variant_effect = pd.concat([ref_df, alt_df], ignore_index=True)

# 2) Remove potential duplicates (e.g., if same name appears multiple times with same/different effects)
name_variant_effect = name_variant_effect.drop_duplicates(subset=['name'])

# 3) Merge the metadata with the name/effect table
metadata_filtered = metadata_table.merge(name_variant_effect, on='name', how='inner')


In [41]:
metadata_filtered
metadata_filtered[col_start] = metadata_filtered[col_start].astype(int)
metadata_filtered[col_end] = metadata_filtered[col_end].astype(int)
metadata_filtered[col_info] = metadata_filtered['rounded_variant_effect']

In [42]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]


In [47]:
metadata_filtered

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,rounded_variant_effect
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AAGTGCTGAGATTACAGGCATGAGCCACTGTGCCCAGCAGGGCTGT...,variant,test,NaN,GRCh38,chr1,2288382,2288652,+,"[SNV, SNV, SNV]","[58, 85, 125]","[NC_000001.11:2288440:A:C, NC_000001.11:228846...","[ref, ref, ref]",0.878,0.878
1,cardiac_neuro_cava_random:REF_PRDM16|ENSG00000...,GGCTCTTTGCATGTCTCTGGGAAAACCTGATCAGCTGACCCAAACC...,variant,test,NaN,GRCh38,chr1,3125026,3125296,+,[SNV],[60],[NC_000001.11:3125086:C:T],[ref],-1.011,-1.011
2,cardiac_neuro_cava_random:REF_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[243, 233, 210, 173, 162, 157, 156, 155, 153, ...","[NC_000001.11:17013955:C:T, NC_000001.11:17013...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",0.812,0.812
3,cardiac_neuro_cava_random:REF_ECE1|ENSG0000011...,TCCAATTGACTGGGTAAGCCTCCTGCCATAGTCAAAGGGCTTCATT...,variant,test,NaN,GRCh38,chr1,21341113,21341383,-,[SNV],[210],[NC_000001.11:21341172:C:T],[ref],-1.038,-1.038
4,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,AAGCGAGTCTCACCAGTGATGTCACACATCTGTTGATTGTCTCCTC...,variant,test,NaN,GRCh38,chr1,27650864,27651134,-,"[SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, SNV, ...","[207, 174, 171, 160, 138, 136, 133, 128, 123, ...","[NC_000001.11:27650926:G:T, NC_000001.11:27650...","[ref, ref, ref, ref, ref, ref, ref, ref, ref, ...",-0.973,-0.973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,cardiac_neuro_cava_random:ALT_KCNQ3|ENSG000001...,GTGGGCACTTGGAATTGACTTTTCTCTAGCGTTTATAAGAAGCCAG...,variant,test,NaN,GRCh38,chr8,132175013,132175283,-,[SNV],[121],[NC_000008.11:132175161:A:G],[alt],0.748,0.748
188,cardiac_neuro_cava_random:ALT_RFX3|ENSG0000008...,AGGAAGCATGACTGGGAGGTCTCACGGAAACTTACCATTATGGGGG...,variant,test,NaN,GRCh38,chr9,3528487,3528757,-,[SNV],[43],[NC_000009.12:3528713:G:C],[alt],0.867,0.867
189,cardiac_neuro_cava_random:ALT_GABBR2|ENSG00000...,GATAAGCAGTCCAGCCCCTGAGGGATCTGGAAGGCTCCTCCCAGCT...,variant,test,NaN,GRCh38,chr9,98319080,98319350,-,[SNV],[34],[NC_000009.12:98319315:A:G],[alt],0.821,0.821
190,cardiac_neuro_cava_random:ALT_DMD|ENSG00000198...,TTTGTTTGTTTGTTTGAGACGGAGTCTTGCTCAGTCGCCCAGGCTG...,variant,test,NaN,GRCh38,chrX,31863087,31863357,-,[SNV],[151],[NC_000023.11:31863205:G:T],[alt],1.325,1.325


In [49]:
metadata_filtered_alt.shape[0]
metadata_filtered_ref.shape[0]

92

In [71]:
# remove variant info for reference sequence
metadata_filtered_alt = metadata_filtered.loc[metadata_filtered[col_name].str.contains(":ALT_")].copy()
metadata_filtered_ref = metadata_filtered.loc[metadata_filtered[col_name].str.contains(":REF_")].copy()

# flatten variant pos:
metadata_filtered_alt[col_variant_pos] = metadata_filtered_alt[col_variant_pos].apply(lambda var_pos: var_pos[0])
metadata_filtered_alt[col_variant_class] = metadata_filtered_alt[col_variant_class].apply(lambda var_class: var_class[0])
metadata_filtered_alt[col_SPDI] = metadata_filtered_alt[col_SPDI].apply(lambda var_spdi: var_spdi[0])
metadata_filtered_alt[col_allele] = metadata_filtered_alt[col_allele].apply(lambda allele: allele[0])

metadata_filtered_ref[[col_variant_class, col_variant_pos, col_SPDI]] = pd.NA
metadata_filtered_ref["allele"] = ["ref"]*metadata_filtered_ref.shape[0]

In [72]:
metadata_filtered_processed = pd.concat([metadata_filtered_alt, metadata_filtered_ref], ignore_index=True)
metadata_filtered_processed

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,rounded_variant_effect
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AAGTGCTGAGATTACAGGCATGAGCCACTGTGCCCAGCAGGGCTGT...,variant,test,NaN,GRCh38,chr1,2288382,2288652,+,SNV,125,NC_000001.11:2288507:G:A,alt,0.878,0.878
1,cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000...,GGCTCTTTGCATGTCTCTGGGAAAACCTGATCAGCTGACCCAAACC...,variant,test,NaN,GRCh38,chr1,3125026,3125296,+,SNV,60,NC_000001.11:3125086:C:T,alt,-1.011,-1.011
2,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,SNV,156,NC_000001.11:17014042:C:T,alt,0.812,0.812
3,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,SNV,155,NC_000001.11:17014043:C:T,alt,0.772,0.772
4,cardiac_neuro_cava_random:ALT_ECE1|ENSG0000011...,TCCAATTGACTGGGTAAGCCTCCTGCCATAGTCAAAGGGCTTCATT...,variant,test,NaN,GRCh38,chr1,21341113,21341383,-,SNV,210,NC_000001.11:21341172:C:T,alt,-1.038,-1.038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,cardiac_neuro_cava_random:REF_VPS13B|ENSG00000...,TGGCCCCTACTTAGAGAATCAAAGAGGTTATTTATTCAGATGATGA...,variant,test,NaN,GRCh38,chr8,99905036,99905306,+,NaN,<NA>,NaN,ref,0.741,0.741
188,cardiac_neuro_cava_random:REF_KCNQ3|ENSG000001...,GTGGGCACTTGGAATTGACTTTTCTCTAGCGTTTATAAGAAGCCAG...,variant,test,NaN,GRCh38,chr8,132175013,132175283,-,NaN,<NA>,NaN,ref,0.748,0.748
189,cardiac_neuro_cava_random:REF_RFX3|ENSG0000008...,AGGAAGCATGACTGGGAGGTCTCACGGAAACTTACCATTATGGCGG...,variant,test,NaN,GRCh38,chr9,3528487,3528757,-,NaN,<NA>,NaN,ref,0.867,0.867
190,cardiac_neuro_cava_random:REF_GABBR2|ENSG00000...,GATAAGCAGTCCAGCCCCTGAGGGATCTGGAAGGTTCCTCCCAGCT...,variant,test,NaN,GRCh38,chr9,98319080,98319350,-,NaN,<NA>,NaN,ref,0.821,0.821


In [74]:
# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = sig_variant_bcalm_metadata_df_gnomad_info_sorted_top100.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = metadata_filtered_processed.loc[metadata_filtered_processed[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict
# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = metadata_filtered_processed.loc[metadata_filtered_processed[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict
# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = metadata_filtered_processed.loc[metadata_filtered_processed[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row):
    if hf.is_reference(row[col_allele]):
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        # NOTE: within variant_pos: for reference sequences a array of variant positions need to be added
        row[col_variant_pos] = [int(alt_variant_pos_dict[alt_id]) for alt_id in ref_alt_dict[row[col_name]]]
        row[col_variant_class] = [alt_variant_class_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

metadata_filtered_processed_data = metadata_filtered_processed.apply(add_spdi_values_2_reference, axis=1)
metadata_filtered_processed_data = metadata_filtered_processed_data[interesting_columns].copy()

In [77]:
# function to generate arrays out off the columns
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

In [78]:
metadata_filtered_processed_data = metadata_filtered_processed_data.apply(make_column_arrays, axis = 1)

In [79]:
metadata_filtered_processed_data

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AAGTGCTGAGATTACAGGCATGAGCCACTGTGCCCAGCAGGGCTGT...,variant,test,NaN,GRCh38,chr1,2288382,2288652,+,[SNV],[125],[NC_000001.11:2288507:G:A],[alt],0.878
1,cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000...,GGCTCTTTGCATGTCTCTGGGAAAACCTGATCAGCTGACCCAAACC...,variant,test,NaN,GRCh38,chr1,3125026,3125296,+,[SNV],[60],[NC_000001.11:3125086:C:T],[alt],-1.011
2,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,[SNV],[156],[NC_000001.11:17014042:C:T],[alt],0.812
3,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,[SNV],[155],[NC_000001.11:17014043:C:T],[alt],0.772
4,cardiac_neuro_cava_random:ALT_ECE1|ENSG0000011...,TCCAATTGACTGGGTAAGCCTCCTGCCATAGTCAAAGGGCTTCATT...,variant,test,NaN,GRCh38,chr1,21341113,21341383,-,[SNV],[210],[NC_000001.11:21341172:C:T],[alt],-1.038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,cardiac_neuro_cava_random:REF_VPS13B|ENSG00000...,TGGCCCCTACTTAGAGAATCAAAGAGGTTATTTATTCAGATGATGA...,variant,test,NaN,GRCh38,chr8,99905036,99905306,+,[SNV],[73],[NC_000008.11:99905109:A:G],[ref],0.741
188,cardiac_neuro_cava_random:REF_KCNQ3|ENSG000001...,GTGGGCACTTGGAATTGACTTTTCTCTAGCGTTTATAAGAAGCCAG...,variant,test,NaN,GRCh38,chr8,132175013,132175283,-,[SNV],[121],[NC_000008.11:132175161:A:G],[ref],0.748
189,cardiac_neuro_cava_random:REF_RFX3|ENSG0000008...,AGGAAGCATGACTGGGAGGTCTCACGGAAACTTACCATTATGGCGG...,variant,test,NaN,GRCh38,chr9,3528487,3528757,-,[SNV],[43],[NC_000009.12:3528713:G:C],[ref],0.867
190,cardiac_neuro_cava_random:REF_GABBR2|ENSG00000...,GATAAGCAGTCCAGCCCCTGAGGGATCTGGAAGGTTCCTCCCAGCT...,variant,test,NaN,GRCh38,chr9,98319080,98319350,-,[SNV],[34],[NC_000009.12:98319315:A:G],[ref],0.821


In [ ]:
metadata_filtered_processed_data

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AAGTGCTGAGATTACAGGCATGAGCCACTGTGCCCAGCAGGGCTGT...,variant,test,NaN,GRCh38,chr1,2288382,2288652,+,NaN,NaN,NaN,NaN,0.878
1,cardiac_neuro_cava_random:REF_PRDM16|ENSG00000...,GGCTCTTTGCATGTCTCTGGGAAAACCTGATCAGCTGACCCAAACC...,variant,test,NaN,GRCh38,chr1,3125026,3125296,+,NaN,NaN,NaN,NaN,-1.011
2,cardiac_neuro_cava_random:REF_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,NaN,NaN,NaN,NaN,0.812
3,cardiac_neuro_cava_random:REF_ECE1|ENSG0000011...,TCCAATTGACTGGGTAAGCCTCCTGCCATAGTCAAAGGGCTTCATT...,variant,test,NaN,GRCh38,chr1,21341113,21341383,-,NaN,NaN,NaN,NaN,-1.038
4,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...,AAGCGAGTCTCACCAGTGATGTCACACATCTGTTGATTGTCTCCTC...,variant,test,NaN,GRCh38,chr1,27650864,27651134,-,NaN,NaN,NaN,NaN,-0.973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,cardiac_neuro_cava_random:ALT_KCNQ3|ENSG000001...,GTGGGCACTTGGAATTGACTTTTCTCTAGCGTTTATAAGAAGCCAG...,variant,test,NaN,GRCh38,chr8,132175013,132175283,-,[SNV],[121],[NC_000008.11:132175161:A:G],[alt],0.748
188,cardiac_neuro_cava_random:ALT_RFX3|ENSG0000008...,AGGAAGCATGACTGGGAGGTCTCACGGAAACTTACCATTATGGGGG...,variant,test,NaN,GRCh38,chr9,3528487,3528757,-,[SNV],[43],[NC_000009.12:3528713:G:C],[alt],0.867
189,cardiac_neuro_cava_random:ALT_GABBR2|ENSG00000...,GATAAGCAGTCCAGCCCCTGAGGGATCTGGAAGGCTCCTCCCAGCT...,variant,test,NaN,GRCh38,chr9,98319080,98319350,-,[SNV],[34],[NC_000009.12:98319315:A:G],[alt],0.821
190,cardiac_neuro_cava_random:ALT_DMD|ENSG00000198...,TTTGTTTGTTTGTTTGAGACGGAGTCTTGCTCAGTCGCCCAGGCTG...,variant,test,NaN,GRCh38,chrX,31863087,31863357,-,[SNV],[151],[NC_000023.11:31863205:G:T],[alt],1.325


In [80]:
# # Write DataFrame to TSV file
group_name = "NGN2_top100_high_effect_variants"
output_path = "/home/kisa/coding/80K_MPRA/collaborations/ahituv_top100_highest_effect_variants_NGN2"

metadata_filtered_processed_data[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0

Check the file for the lowest and highest variant position

In [1]:
import pandas as pd
variant_effect_input_path = "/home/kisa/coding/80K_MPRA/collaborations/ahituv_top100_highest_effect_variants_NGN2/NGN2_top100_high_effect_variants.metadata.tsv.gz"
variant_effect_input_df = pd.read_csv(variant_effect_input_path, sep="\t", low_memory=False)

In [5]:
# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    variant_effect_input_df[col] = variant_effect_input_df[col].apply(safe_eval)

In [2]:
variant_effect_input_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AAGTGCTGAGATTACAGGCATGAGCCACTGTGCCCAGCAGGGCTGT...,variant,test,NaN,GRCh38,chr1,2288382,2288652,+,"[""SNV""]",[125],"[""NC_000001.11:2288507:G:A""]","[""alt""]",0.878
1,cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000...,GGCTCTTTGCATGTCTCTGGGAAAACCTGATCAGCTGACCCAAACC...,variant,test,NaN,GRCh38,chr1,3125026,3125296,+,"[""SNV""]",[60],"[""NC_000001.11:3125086:C:T""]","[""alt""]",-1.011
2,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,"[""SNV""]",[156],"[""NC_000001.11:17014042:C:T""]","[""alt""]",0.812
3,cardiac_neuro_cava_random:ALT_SDHB|ENSG0000011...,TCCGTGGGTTCCCAAGAGAGCCACAGCCTAGAAGGGAAGCTGTTCA...,variant,test,NaN,GRCh38,chr1,17013929,17014199,-,"[""SNV""]",[155],"[""NC_000001.11:17014043:C:T""]","[""alt""]",0.772
4,cardiac_neuro_cava_random:ALT_ECE1|ENSG0000011...,TCCAATTGACTGGGTAAGCCTCCTGCCATAGTCAAAGGGCTTCATT...,variant,test,NaN,GRCh38,chr1,21341113,21341383,-,"[""SNV""]",[210],"[""NC_000001.11:21341172:C:T""]","[""alt""]",-1.038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,cardiac_neuro_cava_random:REF_VPS13B|ENSG00000...,TGGCCCCTACTTAGAGAATCAAAGAGGTTATTTATTCAGATGATGA...,variant,test,NaN,GRCh38,chr8,99905036,99905306,+,"[""SNV""]",[73],"[""NC_000008.11:99905109:A:G""]","[""ref""]",0.741
188,cardiac_neuro_cava_random:REF_KCNQ3|ENSG000001...,GTGGGCACTTGGAATTGACTTTTCTCTAGCGTTTATAAGAAGCCAG...,variant,test,NaN,GRCh38,chr8,132175013,132175283,-,"[""SNV""]",[121],"[""NC_000008.11:132175161:A:G""]","[""ref""]",0.748
189,cardiac_neuro_cava_random:REF_RFX3|ENSG0000008...,AGGAAGCATGACTGGGAGGTCTCACGGAAACTTACCATTATGGCGG...,variant,test,NaN,GRCh38,chr9,3528487,3528757,-,"[""SNV""]",[43],"[""NC_000009.12:3528713:G:C""]","[""ref""]",0.867
190,cardiac_neuro_cava_random:REF_GABBR2|ENSG00000...,GATAAGCAGTCCAGCCCCTGAGGGATCTGGAAGGTTCCTCCCAGCT...,variant,test,NaN,GRCh38,chr9,98319080,98319350,-,"[""SNV""]",[34],"[""NC_000009.12:98319315:A:G""]","[""ref""]",0.821


In [8]:
variants = variant_effect_input_df["variant_pos"].apply(lambda x: x[0] if isinstance(x, list) else x)
print(variants.min(), variants.max())

26 243


In [6]:
type(variant_effect_input_df["variant_pos"].to_list()[0])

list